In [1]:
import os

from google import genai

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_community.retrievers import BM25Retriever

from langchain_classic.retrievers import EnsembleRetriever

C:\Users\Dell\AppData\Local\Temp\ipykernel_12016\479031126.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Load PDFs**

In [2]:
pdf_folder = "all-pdfs"

documents = []

for file in os.listdir(pdf_folder):

    if file.endswith(".pdf"):

        loader = PyPDFLoader(os.path.join(pdf_folder, file))

        documents.extend(loader.load())

print("Total Pages:", len(documents))

Total Pages: 121


**Split-Documents**

In [3]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=700,

    chunk_overlap=150

)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 621


**Create embeddings**

In [4]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Create vector database (FAISS)**

In [5]:
vector_db = FAISS.from_documents(

    chunks,

    embeddings

)

print("Vector Database Created!")

Vector Database Created!


**Create Hybrid retriever (FAIS, BM25, Ensemble)**

In [6]:
vector_retriever = vector_db.as_retriever(

    search_kwargs={"k":2}

)

bm25 = BM25Retriever.from_documents(chunks)

bm25.k = 2

ensemble_retriever = EnsembleRetriever(

    retrievers=[

        bm25,

        vector_retriever

    ],

    weights=[0.5,0.5]

)

print("Hybrid Retriever Ready!")

Hybrid Retriever Ready!


**Create Evaluation Fucntion**

In [7]:
def retrieve_documents(query):

    docs = ensemble_retriever.invoke(query)

    return docs

**Test Retriever**

In [19]:
#query = "Attention Is All You Need Transformer architecture"
query = "Transformer architecture?"
docs = retrieve_documents(query)

for i, doc in enumerate(docs,1):

    print("="*60)

    print(f"Document {i}")

    print()
    
    print(doc.page_content)

Document 1

Table 4: The Transformer generalizes well to English constituency parsing (Results are on Section 23
of WSJ)
Parser Training WSJ 23 F1
Vinyals & Kaiser el al. (2014) [37] WSJ only, discriminative 88.3
Petrov et al. (2006) [29] WSJ only, discriminative 90.4
Zhu et al. (2013) [40] WSJ only, discriminative 90.4
Dyer et al. (2016) [8] WSJ only, discriminative 91.7
Transformer (4 layers) WSJ only, discriminative 91.3
Zhu et al. (2013) [40] semi-supervised 91.3
Huang & Harper (2009) [14] semi-supervised 91.3
McClosky et al. (2006) [26] semi-supervised 92.1
Vinyals & Kaiser el al. (2014) [37] semi-supervised 92.1
Transformer (4 layers) semi-supervised 92.7
Luong et al. (2015) [23] multi-task 93.0
Document 2

Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and

## Optimization Conclusion

I experimented with different values of `chunk_size` and `k`. Reducing `chunk_size` from 1000 to 700 and lowering `k` from 3 to 2 reduced the total number of retrieved documents, but the overall retrieval relevance remained similar. Some Gemini chunks were still retrieved because the embedding model considers BERT and Gemini semantically related due to their shared Transformer and language-model concepts. More advanced techniques such as reranking, metadata filtering, or stronger embedding models can further improve retrieval quality.

**Manual Evaluation**

# Manual RAG Evaluation

| Query | Retrieved Correct Document? | Relevant? | Notes |
|--------|----------------------------|-----------|-------|
| What is BERT? | ✅ Yes | ⭐⭐⭐⭐⭐ | Retrieved BERT explanation |
| Who developed Gemini? | ✅ Yes | ⭐⭐⭐⭐⭐ | Retrieved Gemini paper |
| What is FAISS? | ✅ Yes | ⭐⭐⭐⭐ | Correct definition |
| What is Python? | ❌ No | ⭐ | Not available in PDFs |
| Explain Transformer | ✅ Yes | ⭐⭐⭐⭐⭐ | Retrieved Attention paper |

In [16]:
query = "Transformer architecture"

results = vector_db.similarity_search_with_score(query, k=10)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nRank {i}")
    print("Score:", score)
    print("Source:", doc.metadata["source"])
    print(doc.page_content[:200])


Rank 1
Score: 0.9658697
Source: all-pdfs\attension-is-all-u-need.pdf
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, 

Rank 2
Score: 1.0472846
Source: all-pdfs\attension-is-all-u-need.pdf
model by multiplying the training time, the number of GPUs used, and an estimate of the sustained
single-precision floating-point capacity of each GPU 5.
6.2 Model Variations
To evaluate the importanc

Rank 3
Score: 1.0598912
Source: all-pdfs\attension-is-all-u-need.pdf
The Transformer allows for significantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
2 Backgr

Rank 4
Score: 1.1253033
Source: all-pdfs\bert.pdf
A distinctive feature of BERT is its uniﬁed ar-
chitecture across different tasks. There is mini-
mal difference between the pre-trained

**Build Context**

In [20]:
context = "\n\n".join(

    [doc.page_content for doc in docs]

)

**Create Prompt**

In [21]:
prompt = f"""
You are an AI assistant.

Answer ONLY using the context below.

If the answer is not available, reply:

"I don't know based on the provided context."

Keep the answer concise (2-4 sentences).

Context:
{context}

Question:
{query}

Answer:
"""

**Gemini API Key**

In [22]:
client = genai.Client(
    api_key="your_api_key"
)

response = client.models.generate_content(

    model="gemini-2.5-flash",

    contents=prompt

)

print(response.text)

The Transformer architecture utilizes stacked self-attention and point-wise, fully connected layers for both its encoder and decoder. The encoder is composed of N = 6 identical layers, each containing two sub-layers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network. Residual connections and layer normalization are applied around each sub-layer.


**Cretate ChatBot Function**

In [29]:
def ask_question(question):

    docs = ensemble_retriever.invoke(question)

    context = "\n\n".join(

        [doc.page_content for doc in docs]

    )

    prompt = f"""
    You are an AI assistant.

    Answer ONLY using the context below.

    If the answer is not available, reply:

    "I don't know based on the provided context."

    Keep your answer concise (15-40 sentences).

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return response.text

    except Exception as e:
        print(f"Error: {e}")   # Helpful for debugging
        return "Gemini server is currently busy. Please try again later."

**Interactive ChatBot**

In [30]:
while True:

    question = input("You: ")

    if question.lower() == "exit":

        print("Goodbye!")

        break

    answer = ask_question(question)

    print("\nAssistant:\n")

    print(answer)

    print("-"*60)

You:  Pre-Training Dataset



Assistant:

Gemini models are trained on a pre-training dataset that is both multimodal and multilingual. This dataset incorporates diverse data sources, including web documents, books, code, and also includes image, audio, and video data. The SentencePiece tokenizer (Kudo and Richardson, 2018) is used, and its training on a large sample of the entire training corpus improves the inferred vocabulary and subsequent model performance. Safety filtering is applied to remove harmful content, aligning with established policies. To maintain evaluation integrity, any evaluation data potentially present in the training corpus is identified and removed prior to its use for training. The final data mixtures and weights were determined through ablations conducted on smaller models. Training is staged to dynamically alter the mixture composition, specifically by increasing the weight of domain-relevant data towards the end of training. Data quality is considered an important factor for highly-perf

You:  Evaluation


Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 39.35034749s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'quota

You:  exit


Goodbye!
